# Interactive Algorithm Comparison Dashboard

Click legend entries to toggle algorithms. Double-click to isolate one.

In [5]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

run = 'a_20260402_191005'
# run = None  # Set to None to auto-detect the latest CSV file in results/

csv_file = f'../results/{run}/summary_{run}.csv' if run else None

results_candidates = [Path('results'), Path('../results')]
results_dir = next((p for p in results_candidates if p.exists() and p.is_dir()), None)
if results_dir is None:
    searched = [str(p.resolve()) for p in results_candidates]
    raise FileNotFoundError(f"Results folder not found. Searched: {searched}")

if csv_file is None:
    csv_candidates = sorted(results_dir.rglob('*.csv'))
    if not csv_candidates:
        raise FileNotFoundError(f"No .csv files found inside {results_dir.resolve()}")
    csv_file = str(csv_candidates[0])

print(f"Using CSV file: {csv_file}")
df = pd.read_csv(csv_file)

required_columns = ['instance', 'algorithm', 'objective_mean', 'exec_time_mean']
missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

best_objectives_candidates = [
    Path('best_solutions') / 'best_objectives.csv',
    Path('../best_solutions') / 'best_objectives.csv',
]
best_objectives_path = next((p for p in best_objectives_candidates if p.exists()), None)
if best_objectives_path is None:
    searched = [str(p.resolve()) for p in best_objectives_candidates]
    raise FileNotFoundError(f"Best objectives file not found. Searched: {searched}")

best_df = pd.read_csv(best_objectives_path)
if 'instance' not in best_df.columns or 'best_objective' not in best_df.columns:
    raise ValueError("best_objectives.csv must contain columns: instance, best_objective")

if 'dataset' in df.columns and 'dataset' in best_df.columns:
    merge_columns = ['dataset', 'instance']
else:
    merge_columns = ['instance']

df = df.merge(best_df[merge_columns + ['best_objective']], on=merge_columns, how='left')
ordered_columns = [col for col in df.columns if col != 'best_objective'] + ['best_objective']
df = df[ordered_columns]

df.head()

Using CSV file: ../results/a_20260402_191005/summary_a_20260402_191005.csv


,dataset,algorithm,instance,total_runs,feasible_runs,infeasible_runs,timed_out_runs,objective_mean,objective_median,objective_mode,...,aisles_variance,aisles_std_dev,exec_time_mean,exec_time_median,exec_time_mode,exec_time_min,exec_time_max,exec_time_variance,exec_time_std_dev,best_objective
0,a,aisle_cluster_expansion,instance_0001.txt,30,30,0,0,15.00,15.00,15.00,...,0.0,0.0,0.141049,0.138213,0.110738,0.108256,0.201224,5.544191e-04,0.023546,15.000
1,a,aisle_cluster_expansion,instance_0002.txt,30,30,0,0,2.00,2.00,2.00,...,0.0,0.0,0.000478,0.000448,0.000493,0.000409,0.001053,1.260818e-08,0.000112,2.000
2,a,aisle_cluster_expansion,instance_0003.txt,30,30,0,0,8.75,8.75,8.75,...,0.0,0.0,0.268369,0.270527,0.219854,0.216268,0.313109,1.225278e-03,0.035004,12.000
3,a,aisle_cluster_expansion,instance_0004.txt,30,30,0,0,3.50,3.50,3.50,...,0.0,0.0,0.022784,0.022096,0.025374,0.016192,0.034029,2.230135e-05,0.004722,3.500
4,a,aisle_cluster_expansion,instance_0005.txt,30,29,1,1,143.50,143.50,143.50,...,0.0,0.0,15.077856,16.041228,7.705076,7.705076,19.037742,6.778748e+00,2.603603,177.875


In [6]:
# Objective Mean as % of Best Objective

objective_plot = (
    df.pivot_table(
        index='instance',
        columns='algorithm',
        values='objective_mean',
        aggfunc='mean'
    )
    .sort_index()
    .fillna(0)
)

best_objective_by_instance = (
    df[['instance', 'best_objective']]
    .drop_duplicates(subset='instance')
    .set_index('instance')['best_objective']
)

objective_pct_plot = objective_plot.div(best_objective_by_instance, axis=0) * 100
objective_pct_plot = objective_pct_plot.replace([float('inf'), -float('inf')], 0).fillna(0)

fig = go.Figure()
for algo in objective_pct_plot.columns:
    fig.add_trace(go.Bar(
        name=algo,
        x=objective_pct_plot.index,
        y=objective_pct_plot[algo],
        hovertemplate=f'<b>{algo}</b><br>Instance: %{{x}}<br>%{{y:.1f}}% of best<extra></extra>'
    ))

fig.add_hline(y=100, line_dash='dash', line_color='black', annotation_text='best (100%)')

fig.update_layout(
    barmode='group',
    title='Objective Mean by Instance and Algorithm (% of Best Objective)',
    xaxis_title='Instance',
    yaxis_title='% of best_objective',
    legend_title='Algorithm',
    height=700,
    xaxis_tickangle=-45,
)
fig.show()

In [7]:
# Execution Time Mean

exec_time_plot = (
    df.pivot_table(
        index='instance',
        columns='algorithm',
        values='exec_time_mean',
        aggfunc='mean'
    )
    .sort_index()
    .fillna(0)
)

fig = go.Figure()
for algo in exec_time_plot.columns:
    fig.add_trace(go.Bar(
        name=algo,
        x=exec_time_plot.index,
        y=exec_time_plot[algo],
        hovertemplate=f'<b>{algo}</b><br>Instance: %{{x}}<br>%{{y:.4f}}s<extra></extra>'
    ))

fig.update_layout(
    barmode='group',
    title='Execution Time Mean by Instance and Algorithm',
    xaxis_title='Instance',
    yaxis_title='exec_time_mean (s)',
    legend_title='Algorithm',
    height=600,
    xaxis_tickangle=-45,
)
fig.show()

In [8]:
# Rankings

ranking_dataset_param = None  # e.g., 'a', 'b', 'x' or None for all datasets

ranking_base = df[['instance', 'algorithm', 'objective_mean', 'exec_time_mean']].copy()
if ranking_dataset_param is not None and 'dataset' in df.columns:
    ranking_base = df[df['dataset'] == ranking_dataset_param][
        ['instance', 'algorithm', 'objective_mean', 'exec_time_mean']
    ].copy()

if ranking_base.empty:
    raise ValueError(f'No rows found for dataset={ranking_dataset_param!r}')


def build_final_ranking(base_df: pd.DataFrame, metric_column: str, ascending: bool) -> pd.DataFrame:
    metric_df = base_df.dropna(subset=[metric_column]).copy()
    if metric_df.empty:
        raise ValueError(f'No non-null values found for metric {metric_column!r}.')

    metric_df['N'] = metric_df.groupby('instance')['algorithm'].transform('nunique')
    metric_df['rank_class'] = metric_df.groupby('instance')[metric_column].rank(
        method='min', ascending=ascending
    ).astype(int)
    metric_df['score'] = metric_df['N'] - metric_df['rank_class']

    final_ranking = (
        metric_df.groupby('algorithm', as_index=False)['score']
        .agg(total_score='sum', average_score='mean', instances_count='count')
        .sort_values(['total_score', 'average_score', 'algorithm'], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    final_ranking['final_rank'] = final_ranking['total_score'].rank(
        method='min', ascending=False
    ).astype(int)

    return final_ranking[
        ['final_rank', 'algorithm', 'total_score', 'average_score', 'instances_count']
    ]


objective_ranking = build_final_ranking(
    ranking_base, metric_column='objective_mean', ascending=False
)
exec_time_ranking = build_final_ranking(
    ranking_base, metric_column='exec_time_mean', ascending=True
)

print(f'Dataset filter for ranking: {ranking_dataset_param!r}')
print('\nOBJECTIVE_MEAN - FINAL RANKING')
display(objective_ranking)
print('\nEXEC_TIME_MEAN - FINAL RANKING')
display(exec_time_ranking)

Dataset filter for ranking: None

OBJECTIVE_MEAN - FINAL RANKING


,final_rank,algorithm,total_score,average_score,instances_count
0,1,lagrangian_relaxation,433,21.65,20
1,2,smaller_multi,412,20.60,20
2,3,aisle_cluster_expansion,385,19.25,20
3,4,smaller,362,18.10,20
4,5,similar_bigger_multi,329,16.45,20
5,6,column_generation,320,16.00,20
6,6,lp_aisle_focus,320,16.00,20
7,8,random_multi,319,15.95,20
8,9,diff_multi,315,15.75,20
9,10,similar_smaller_multi,314,15.70,20



EXEC_TIME_MEAN - FINAL RANKING


,final_rank,algorithm,total_score,average_score,instances_count
0,1,aisle_first_random,435,21.75,20
1,2,aisle_first_desc,421,21.05,20
2,3,aisle_first_asc,416,20.80,20
3,4,smaller,404,20.20,20
4,5,random,389,19.45,20
5,6,greedy_set_cover,385,19.25,20
6,7,min_aisle_cover,357,17.85,20
7,8,diff_smaller,317,15.85,20
8,9,similar,311,15.55,20
9,10,smaller_multi,284,14.20,20
